In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML, display_html
import seaborn as sns

from pygmt_helper import plotting

import nn_gmm as nng
import ml_tools as mlt

PLOT_IMS = ["pSA_0.01", "pSA_0.1", "pSA_0.5", "pSA_1.0", "pSA_3.0", "pSA_10.0"]

In [ ]:
results_dir = "/Users/claudy/dev/work/data/nn_gmm/results/1029_1349_cv_v1_25Epochs_3E4Folds_1000Sites"

In [ ]:
# Injected Parameters
results_dir = "/home/claudy/dev/work/data/nn_gmm/results/1101_1435_cv_v1_25Epochs_6E6Folds"


In [ ]:
results_dir = Path(results_dir)
run_config = nng.nn_gmm.load_config(results_dir / "run_config.yaml")

print(f"Results Directory: {results_dir}")

In [ ]:
val_df = pd.read_parquet(results_dir / "val_results.parquet").sort_index()
val_df.loc[:, run_config.pred_mean_keys] = np.exp(val_df[run_config.pred_mean_keys])

In [ ]:
with nng.DuckIMDB(run_config.imdb_ffp, readonly=True) as imdb:
    event_df = imdb.get_event_df()
    site_df = imdb.get_site_df()
    # sim_df = imdb.get_im_data(run_config.ims, val_df.index.values).sort_index()
    record_info_df = imdb.get_record_info_df(record_int_ids=val_df.index.values).sort_index()
    record_info_df["site_event_int_id"] = nng.utils.get_site_event_int_id(record_info_df.site_int_id.values, 
                                                                          record_info_df.event_int_id.values)
    # event_site_df = imdb.get_site_event_df(site_event_int_ids=record_info_df.site_event_int_id.values)
    # rel_df = imdb.get_rel_df()
# assert sim_df.index.equals(val_df.index)
assert record_info_df.index.equals(val_df.index)

In [ ]:
val_df["event_id"] = record_info_df["event_id"]
val_df["event_int_id"] = record_info_df["event_int_id"]
val_df["magnitude"] = event_df.loc[val_df.event_int_id, "magnitude"].values

In [ ]:
loss_columns = [col for col in val_df.columns if col.endswith("_loss")]
val_df["mean_loss"] = val_df[loss_columns].mean(axis=1)

# loss_df = val_df[loss_columns].astype(np.float64).rename(columns=lambda x: x.replace("_loss", ""))
# record_loss = loss_df.mean(axis=1).to_frame("mean_loss")
# record_loss["site_int_id"] = record_info_df.loc[record_loss.index, "site_int_id"]
# record_loss["event_id"] = record_info_df.loc[record_loss.index, "event_id"]
# record_loss["rel_int_id"] = record_info_df.loc[record_loss.index, "rel_int_id"]

In [ ]:
event_df["cv_iters"] = ""
for cur_int_id in event_df.index.values:
    mask = val_df.event_int_id == cur_int_id
    event_df.at[cur_int_id, "cv_iters"] = "_".join(np.sort(val_df.loc[mask].cv_iter.unique().astype(str)))

event_df = event_df.astype({"cv_iters": "category"})

In [ ]:
val_df["event_fold"] = event_df.loc[val_df.event_int_id, "cv_iters"].values

## Event Loss 
Sorted by magnitude (increasing left to right)

In [ ]:
cur_event_int_ids = event_df.loc[event_df.index.isin(val_df.event_int_id.unique())].sort_values("magnitude", ascending=False).head(15).index.values
cur_event_ids = event_df.loc[cur_event_int_ids, "event_id"].values

cur_df = val_df.loc[val_df.event_int_id.isin(cur_event_int_ids), ["event_id", "mean_loss", "magnitude", "event_fold"]].copy().astype({"event_id": "str", "event_fold": "str"})

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))

sns.stripplot(data=cur_df, x="event_id", y="mean_loss", order=cur_event_ids[::-1], ax=ax, hue="event_fold")
# sns.boxplot(data=cur_df, x="event_id", y="mean_loss", order=cur_event_ids[::-1], ax=ax, whis=(1, 99))

ax.axhline(val_df["mean_loss"].median(), color="k", linestyle="--", label="Median Loss (All)")
ax.axhline(val_df["mean_loss"].quantile(0.16), color="green", linestyle="--", label="16th/84th Percentile")
ax.axhline(val_df["mean_loss"].quantile(0.84), color="green", linestyle="--")
ax.axhline(val_df["mean_loss"].quantile(0.01), color="orange", linestyle="--", label="1st/99th Percentile")
ax.axhline(val_df["mean_loss"].quantile(0.99), color="orange", linestyle="--")
ax.axhline(val_df["mean_loss"].quantile(0.001), color="red", linestyle="--", label="0.1st/99.9th Percentile")
ax.axhline(val_df["mean_loss"].quantile(0.999), color="red", linestyle="--")

ax.legend()
ax.xaxis.set_tick_params(rotation=90)
ax.set_ylim(-10, 200)

fig.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))

sns.boxplot(data=cur_df, x="event_id", y="mean_loss", order=cur_event_ids[::-1], ax=ax, whis=(1, 99), hue="event_fold")

ax.axhline(val_df["mean_loss"].median(), color="k", linestyle="--", label="Median Loss (All)")
ax.axhline(val_df["mean_loss"].quantile(0.16), color="green", linestyle="--", label="16th/84th Percentile")
ax.axhline(val_df["mean_loss"].quantile(0.84), color="green", linestyle="--")
ax.axhline(val_df["mean_loss"].quantile(0.01), color="orange", linestyle="--", label="1st/99th Percentile")
ax.axhline(val_df["mean_loss"].quantile(0.99), color="orange", linestyle="--")
ax.axhline(val_df["mean_loss"].quantile(0.001), color="red", linestyle="--", label="0.1st/99.9th Percentile")
ax.axhline(val_df["mean_loss"].quantile(0.999), color="red", linestyle="--")

ax.legend()
ax.xaxis.set_tick_params(rotation=90)
ax.set_ylim(-10, 200)

fig.tight_layout()

Whiskers are 1st & 99th percentile.